<a href="https://colab.research.google.com/github/NyxLumen/Encephlo/blob/main/Notebooks/SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import zipfile
import os

# 1. Mount Google Drive
print("🔄 Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define Paths
zip_path = "/content/drive/My Drive/mri_train.zip"
extract_path = "/content/dataset"

# 3. Verify & Extract
if os.path.exists(zip_path):
    print(f"✅ Found zip file at: {zip_path}")
    print(f"📂 Extracting to: {extract_path} ...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print("🎉 SUCCESS! Data is ready.")
else:
    print(f"❌ ERROR: Could not find '{zip_path}'")

🔄 Mounting Google Drive...
Mounted at /content/drive
✅ Found zip file at: /content/drive/My Drive/mri_train.zip
📂 Extracting to: /content/dataset ...
🎉 SUCCESS! Data is ready.


In [2]:
import os
import numpy as np
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import torch
import tensorflow as tf
from PIL import Image
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import joblib
from tqdm.auto import tqdm

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
DATASET_DIR = "/content/dataset/Training"

VIT_PATH = "/content/vit_feature_extractor.pt"
DENSENET_PATH = "/content/densenet_headless.keras" # Updated to .keras
SVM_OUTPUT_PATH = "/content/svm_fusion_weights.pkl"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Hardware Accelerator: {device}")

# ─────────────────────────────────────────────
# 1. LOAD MODELS
# ─────────────────────────────────────────────
print("🚀 Loading Feature Extractors into VRAM...")

# PyTorch ViT -> GPU
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
vit_model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224-in21k", num_labels=4)
vit_model.vit.load_state_dict(torch.load(VIT_PATH, map_location=device))
vit_model.to(device)
vit_model.eval()

# TensorFlow DenseNet (Decapitated) -> .keras format
tf_model = tf.keras.models.load_model(DENSENET_PATH, compile=False)
headless_tf = tf.keras.Model(inputs=tf_model.input, outputs=tf_model.layers[-2].output)

# ─────────────────────────────────────────────
# 2. EXTRACT FEATURES (GPU ACCELERATED)
# ─────────────────────────────────────────────
classes = ['glioma', 'meningioma', 'notumor', 'pituitary']
X_features = []
y_labels = []

print(f"📂 Scanning dataset at: {DATASET_DIR}")

for label_idx, class_name in enumerate(classes):
    class_dir = os.path.join(DATASET_DIR, class_name)
    if not os.path.exists(class_dir):
        print(f"⚠️ Warning: Folder {class_dir} missing!")
        continue

    images = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Processing {class_name} ({len(images)} images)...")

    # 400 images per class for robust SVM boundaries
    for img_name in tqdm(images[:400]):
        img_path = os.path.join(class_dir, img_name)

        try:
            # --- ViT Extraction ---
            img_pil = Image.open(img_path).convert("RGB")
            pt_inputs = processor(images=img_pil, return_tensors="pt")
            pt_inputs = {k: v.to(device) for k, v in pt_inputs.items()}

            with torch.no_grad():
                vit_feat = vit_model.vit(**pt_inputs).last_hidden_state[:, 0, :].cpu().numpy()

            # --- DenseNet Extraction ---
            tf_img = tf.io.read_file(img_path)
            tf_img = tf.image.decode_jpeg(tf_img, channels=3)
            tf_img = tf.image.resize(tf_img, (224, 224)) / 255.0
            tf_img = tf.expand_dims(tf_img, axis=0)
            tf_feat = headless_tf.predict(tf_img, verbose=0)

            # --- Fusion ---
            fusion_vector = np.concatenate([tf_feat, vit_feat], axis=1)

            X_features.append(fusion_vector[0])
            y_labels.append(label_idx)

        except Exception as e:
            continue

X_features = np.array(X_features)
y_labels = np.array(y_labels)

# ─────────────────────────────────────────────
# 3. TRAIN SVM
# ─────────────────────────────────────────────
print(f"\n🧠 Training SVM on {len(X_features)} samples of dimension {X_features.shape[1]}...")
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_features, y_labels)

predictions = svm.predict(X_features)
acc = accuracy_score(y_labels, predictions)
print(f"🏆 SVM Training Accuracy: {acc * 100:.2f}%")

joblib.dump(svm, SVM_OUTPUT_PATH)
print(f"✅ SUCCESS! Download {SVM_OUTPUT_PATH} from the files tab.")

🚀 Hardware Accelerator: cuda
🚀 Loading Feature Extractors into VRAM...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

...

Processing notumor (1595 images)...


  0%|          | 0/400 [00:00<?, ?it/s]

Processing pituitary (1457 images)...


  0%|          | 0/400 [00:00<?, ?it/s]


🧠 Training SVM on 1600 samples of dimension 1792...
🏆 SVM Training Accuracy: 100.00%
✅ SUCCESS! Download /content/svm_fusion_weights.pkl from the files tab.
